# <font size=6><b>Lec05. [실습] 유사한 파일 탐색 

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings(action='ignore')

# ----------------------------------------------------------- 토큰화
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
# ----------------------------------------------------------- 유사도
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


#-------------------- 차트 관련 속성 (한글처리, 그리드) -----------
plt.rcParams['font.family']= 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

#-------------------- 주피터 , 출력결과 넓이 늘리기 ---------------
# from IPython.core.display import display, HTML
from IPython.display import display, HTML
display(HTML("<style>.container{width:100% !important;}</style>"))
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('max_colwidth', None)

In [39]:
import os

folder = "C:/IT/workspace_python/LLM/lec05_data"

flist = []
for fname in os.listdir(folder):
    if fname.endswith(".txt"):
        path_filename = os.path.join(folder, fname).replace("\\", "/")
        print(path_filename)
        with open(path_filename, "r", encoding="utf-8") as f:
            #print(fname , f.read()[:80])
            flist.append([path_filename, f.read()])
        
np.array(flist).shape

C:/IT/workspace_python/LLM/lec05_data/a.txt
C:/IT/workspace_python/LLM/lec05_data/b.txt
C:/IT/workspace_python/LLM/lec05_data/c.txt
C:/IT/workspace_python/LLM/lec05_data/d.txt
C:/IT/workspace_python/LLM/lec05_data/e.txt
C:/IT/workspace_python/LLM/lec05_data/f.txt
C:/IT/workspace_python/LLM/lec05_data/g.txt
C:/IT/workspace_python/LLM/lec05_data/h.txt


(8, 2)

In [40]:
df = pd.DataFrame(flist, columns=["fname", "content"])
df.head(1)

,fname,content
0,C:/IT/workspace_python/LLM/lec05_data/a.txt,"유사도(Similarity)는 두 데이터 객체(문서, 벡터 등)가 얼마나 비슷한지 정량적으로 수치화한 지표로, 주로 0에서 1 사이(또는 -1~1) 값으로 표현됩니다. 높을수록(1에 가까울수록) 두 데이터가 유사하며, KCI 문헌 유사도 검사 서비스나 꼬맨틀 등에서 데이터 분석, 표절 검사, 추천 시스템의 핵심 기술로 사용됩니다. \n\n1. 주요 유사도 측정 방식\n코사인 유사도 (Cosine Similarity): 두 벡터 간의 사잇각을 이용하여 유사도를 측정하는 방식입니다. 방향의 유사성을 중요하게 보며, 각도가 좁을수록(0도에 가까울수록) 유사도가 1에 가까워집니다.\n유클리드 거리 (Euclidean Distance): 두 점 사이의 기하학적 거리를 계산하여 유사도를 측정합니다. 거리가 작을수록 유사도가 높다고 판단하며, 비유사도를 측정하는 방식입니다."


In [41]:
df.shape

(8, 2)

In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   fname    8 non-null      object
 1   content  8 non-null      object
dtypes: object(2)
memory usage: 256.0+ bytes


<font size=5><b>------------------------------------------------------------------------

# <b>중복제거 

In [43]:
print(df.shape)
df = df.drop_duplicates(subset=['fname']).reset_index(drop=True)
print(df.shape)

(8, 2)
(8, 2)


In [44]:
df.head()

,fname,content
0,C:/IT/workspace_python/LLM/lec05_data/a.txt,"유사도(Similarity)는 두 데이터 객체(문서, 벡터 등)가 얼마나 비슷한지 정량적으로 수치화한 지표로, 주로 0에서 1 사이(또는 -1~1) 값으로 표현됩니다. 높을수록(1에 가까울수록) 두 데이터가 유사하며, KCI 문헌 유사도 검사 서비스나 꼬맨틀 등에서 데이터 분석, 표절 검사, 추천 시스템의 핵심 기술로 사용됩니다. \n\n1. 주요 유사도 측정 방식\n코사인 유사도 (Cosine Similarity): 두 벡터 간의 사잇각을 이용하여 유사도를 측정하는 방식입니다. 방향의 유사성을 중요하게 보며, 각도가 좁을수록(0도에 가까울수록) 유사도가 1에 가까워집니다.\n유클리드 거리 (Euclidean Distance): 두 점 사이의 기하학적 거리를 계산하여 유사도를 측정합니다. 거리가 작을수록 유사도가 높다고 판단하며, 비유사도를 측정하는 방식입니다."
1,C:/IT/workspace_python/LLM/lec05_data/b.txt,"데이터 과학/AI: 군집화(Clustering), 분류(Classification), 데이터 라벨링 및 AI 모델 학습/진단에 활용됩니다.\n문서 및 코드 유사도: KCI(한국학술지인용색인) 등을 활용해 문서 간 유사성을 측정하여 표절을 검사하거나, 파이썬 등 코드 도용을 방지하기 위한 도구로 활용됩니다 KCI 문헌 유사도 검사 서비스.\n비유사도(Dissimilarity): 데이터 간의 거리를 나타내며, 0에 가까울수록 유사도가 높고 비유사도가 낮다는 것을 의미합니다. \n"
2,C:/IT/workspace_python/LLM/lec05_data/c.txt,전쟁은 국가 또는 정치 집단 사이의 폭력이나 무력을 사용하는 상태 또는 행동을 말한다. 특별히 둘 이상의 국가 간에 어떤 목적을 두고 수행되는 싸움이다. 그러나 독일 농민전쟁 같은 내전도 전쟁이다. 치열한 경쟁이나 혼란을 전쟁에 비유하여 말하기도 한다.
3,C:/IT/workspace_python/LLM/lec05_data/d.txt,전쟁은 국가 또는 정치 집단 사이의 폭력이나 무력을 사용하는 상태 또는 행동을 말한다. 특별히 둘 이상의 국가 간에 어떤 목적을 두고 수행되는 싸움이다. 그러나 독일 농민전쟁 같은 내전도 전쟁이다. 치열한 경쟁이나 혼란을 전쟁에 비유하여 말하기도 한다.
4,C:/IT/workspace_python/LLM/lec05_data/e.txt,"머신 러닝(Machine Learning, 기계 학습)은 컴퓨터가 명시적으로 프로그래밍되지 않고도 데이터와 경험을 통해 스스로 패턴을 학습하고 예측 모델을 빌드하여 성능을 개선하는 인공지능(AI)의 하위 분야입니다. 인공지능이 인간의 지능을 모방한다면, 머신 러닝은 데이터를 기반으로 규칙을 스스로 찾아내 의사결정을 내리는 수단입니다. \n핵심 정의 및 특징\n스스로 학습: 데이터를 분석하여 패턴을 찾고, 학습량이 많아질수록 더 빠르고 정확해집니다.\n기반 기술: 데이터와 알고리즘을 활용하여 인간과 유사한 방식으로 문제를 해결합니다.\n데이터 주도: 사람이 규칙을 코딩하는 대신, 기계가 데이터를 통해 직접 규칙을 찾습니다."


# <b>불용어 사전 로드
* 사전  : nltk.stopwards("my_korean")

In [45]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize 
stop_words_list = stopwords.words('my_korean')    #----C:\Users\user\..\stopwords\my_korean.txt
print('불용어 개수 :', len(stop_words_list))
print('불용어 10개 출력 :',stop_words_list[:10])


불용어 개수 : 595
불용어 10개 출력 : ['가', '가까스로', '가령', '각', '각각', '각자', '각종', '갖고말하자면', '같다', '같이']


# <b>명사 추출
* Mecab : 명사추출

In [46]:
from konlpy.tag import Mecab
mecab = Mecab(dicpath=r"C:/mecab/mecab-ko-dic") 
print('Mecab 명사 추출 :'  , mecab.nouns ("열심히 코딩한 당신, 연휴에는 여행을 가봐요")) 

Mecab 명사 추출 : ['코딩', '당신', '연휴', '여행']


In [47]:
clean_content_list = []
for  text in df['content']:
    #print(text)
    word_list = mecab.nouns (text)
    word_list = [w for w in word_list    if len(w) > 1]
    word_list = " ".join(word_list)
    clean_content_list.append(word_list)

clean_content_list[:2]

['유사 데이터 객체 문서 벡터 정량 수치 지표 사이 표현 데이터 유사 문헌 유사 검사 서비스 맨틀 데이터 분석 표절 검사 추천 시스템 핵심 기술 사용 주요 유사 측정 방식 코사인 유사 벡터 사잇각 이용 유사 측정 방식 방향 유사 중요 각도 유사 유클리드 거리 사이 기하학 거리 계산 유사 측정 거리 유사 판단 비유 사도 측정 방식',
 '데이터 과학 군집 분류 데이터 라벨 모델 학습 진단 활용 문서 코드 유사 한국 학술지 인용 색인 활용 문서 유사 측정 표절 검사 파이썬 코드 도용 방지 도구 활용 문헌 유사 검사 서비스 비유 사도 데이터 거리 유사 비유 사도 의미']

In [48]:
np.array(clean_content_list).shape

(8,)

In [49]:
tfidf_vt = TfidfVectorizer(binary=True)
res = tfidf_vt.fit_transform(clean_content_list)  #---------- 1D
#print(tfidf_vt.vocabulary_)

print(res.shape)
print(tfidf_vt.get_feature_names_out())

tfidf_df = pd.DataFrame(res.toarray(), columns=tfidf_vt.get_feature_names_out())
tfidf_df.head(2)

(8, 196)
['가능' '각도' '강화' '개념' '개선' '개입' '객체' '거래' '거리' '검사' '결정' '결합' '경쟁' '경험'
 '계면' '계산' '고정' '과학' '관계' '관련' '구조' '국가' '군집' '규칙' '극대' '금융' '기계' '기반'
 '기술' '기하학' '나무' '내전' '넷플릭스' '농민' '능력' '다층' '대량' '대신' '데이터' '도구' '도용' '독일'
 '동의어' '라벨' '랜덤' '러닝' '로지스틱' '마이닝' '마진' '맞춤' '매출' '매핑' '맨틀' '머신' '메일' '명시'
 '모델' '모방' '목적' '목표' '무력' '문서' '문제' '문헌' '반면' '발전' '방법론' '방식' '방지' '방향'
 '번역' '범주' '벡터' '보상' '보안' '부분집합' '부하' '분류' '분석' '분야' '비유' '비지' '사도' '사람'
 '사물' '사용' '사이' '사잇각' '사진' '상태' '색인' '서비스' '서포트' '선형' '성능' '수단' '수치' '수학'
 '수행' '스무고개' '스팸' '시스템' '신경망' '싸움' '알고리즘' '앙상블' '언어' '얼굴' '업무' '에너지' '여부'
 '연속' '예시' '예측' '용어' '유사' '유클리드' '유튜브' '유형' '음성' '응용' '의미' '의사' '이미지' '이상'
 '이용' '인간' '인공' '인공지능' '인식' '인용' '일반' '입력' '자동' '자율' '작업' '전쟁' '점진' '정답'
 '정량' '정의' '정치' '정확' '종류' '주가' '주도' '주식' '주요' '주행' '중요' '지능' '지도' '지표'
 '진단' '집단' '징후' '차이' '차이점' '최대' '최적' '추천' '추출' '측정' '치열' '컴퓨터' '코드' '코딩'
 '코사인' '콘텐츠' '탐지' '트레이딩' '트리' '특정' '특징' '파이썬' '판단' '패턴' '포레스트' '폭력' '표절'
 '표현' '프로그래밍' '프로그램' '필요' '하위' '학술지' '학습'

,가능,각도,강화,개념,개선,개입,객체,거래,거리,검사,결정,결합,경쟁,경험,계면,계산,고정,과학,관계,관련,구조,국가,군집,규칙,극대,금융,기계,기반,기술,기하학,나무,내전,넷플릭스,농민,능력,다층,대량,대신,데이터,도구,도용,독일,동의어,라벨,랜덤,러닝,로지스틱,마이닝,마진,맞춤,...,주식,주요,주행,중요,지능,지도,지표,진단,집단,징후,차이,차이점,최대,최적,추천,추출,측정,치열,컴퓨터,코드,코딩,코사인,콘텐츠,탐지,트레이딩,트리,특정,특징,파이썬,판단,패턴,포레스트,폭력,표절,표현,프로그래밍,프로그램,필요,하위,학술지,학습,한국,해결,핵심,행동,향상,혼란,확률,활용,회귀
0,0.0,0.190441,0.0,0.0,0.0,0.0,0.190441,0.0,0.159605,0.159605,0.0,0.0,0.0,0.0,0.0,0.190441,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.137726,0.190441,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.095166,0.000000,0.000000,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.120755,0.0,0.190441,0.0,0.0,0.190441,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.159605,0.0,0.159605,0.0,0.0,0.000000,0.0,0.190441,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.190441,0.0,0.0,0.0,0.159605,0.190441,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.159605,0.0,0.0,0.0,0.0,0.000000,0.0
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.180070,0.180070,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.214861,0.0,0.0,0.0,0.0,0.18007,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.107368,0.214861,0.214861,0.0,0.0,0.18007,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.214861,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.180070,0.0,0.0,0.214861,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.214861,0.000000,0.0,0.0,0.0,0.180070,0.000000,0.0,0.0,0.0,0.0,0.214861,0.120595,0.214861,0.0,0.000000,0.0,0.0,0.0,0.0,0.136239,0.0


# <b>유사도 측정
* 카운트 기반 임베딩 : TfidfVectorize
* 유사도 : cosine_similarity

In [50]:
tfidf_cos_matrix = cosine_similarity(tfidf_df , tfidf_df)

In [51]:
tfidf_cos_matrix.shape

(8, 8)

In [87]:
tfidf_cos_df = pd.DataFrame(tfidf_cos_matrix, index=df['fname'] , columns=df['fname'])
tfidf_cos_df.head(3)

fname,C:/IT/workspace_python/LLM/lec05_data/a.txt,C:/IT/workspace_python/LLM/lec05_data/b.txt,C:/IT/workspace_python/LLM/lec05_data/c.txt,C:/IT/workspace_python/LLM/lec05_data/d.txt,C:/IT/workspace_python/LLM/lec05_data/e.txt,C:/IT/workspace_python/LLM/lec05_data/f.txt,C:/IT/workspace_python/LLM/lec05_data/g.txt,C:/IT/workspace_python/LLM/lec05_data/h.txt
fname,,,,,,,,
C:/IT/workspace_python/LLM/lec05_data/a.txt,1.000000,0.270651,0.063541,0.063541,0.097777,0.102904,0.041212,0.094443
C:/IT/workspace_python/LLM/lec05_data/b.txt,0.270651,1.000000,0.023896,0.023896,0.081778,0.094750,0.046496,0.056006
C:/IT/workspace_python/LLM/lec05_data/c.txt,0.063541,0.023896,1.000000,1.000000,0.000000,0.051933,0.017582,0.052103


In [88]:
tfidf_cos_df.loc['C:/IT/workspace_python/LLM/lec05_data/a.txt'].sort_values(ascending=False)

fname
C:/IT/workspace_python/LLM/lec05_data/a.txt    1.000000
C:/IT/workspace_python/LLM/lec05_data/b.txt    0.270651
C:/IT/workspace_python/LLM/lec05_data/f.txt    0.102904
C:/IT/workspace_python/LLM/lec05_data/e.txt    0.097777
C:/IT/workspace_python/LLM/lec05_data/h.txt    0.094443
C:/IT/workspace_python/LLM/lec05_data/c.txt    0.063541
C:/IT/workspace_python/LLM/lec05_data/d.txt    0.063541
C:/IT/workspace_python/LLM/lec05_data/g.txt    0.041212
Name: C:/IT/workspace_python/LLM/lec05_data/a.txt, dtype: float64

In [89]:
search_title = "머신 러닝"
print(df[df['content'].str.contains(search_title, regex=False, na=False)]['content'] [:5] )

4                                                                                                                                                                                                                                           머신 러닝(Machine Learning, 기계 학습)은 컴퓨터가 명시적으로 프로그래밍되지 않고도 데이터와 경험을 통해 스스로 패턴을 학습하고 예측 모델을 빌드하여 성능을 개선하는 인공지능(AI)의 하위 분야입니다. 인공지능이 인간의 지능을 모방한다면, 머신 러닝은 데이터를 기반으로 규칙을 스스로 찾아내 의사결정을 내리는 수단입니다. \n핵심 정의 및 특징\n스스로 학습: 데이터를 분석하여 패턴을 찾고, 학습량이 많아질수록 더 빠르고 정확해집니다.\n기반 기술: 데이터와 알고리즘을 활용하여 인간과 유사한 방식으로 문제를 해결합니다.\n데이터 주도: 사람이 규칙을 코딩하는 대신, 기계가 데이터를 통해 직접 규칙을 찾습니다.
5    머신 러닝의 주요 유형\n지도 학습 (Supervised Learning): 라벨링된(정답이 있는) 데이터를 통해 학습하여 입력값에 대한 예측값(분류/회귀)을 학습합니다.\n비지도 학습 (Unsupervised Learning): 라벨링되지 않은 데이터에서 숨겨진 구조나 패턴을 찾아냅니다.\n강화 학습 (Reinforcement Learning): 행동에 대한 보상을 통해 스스로 최적의 행동 방식을 학습합니다. \n\n머신 러닝 활용 예시\n추천 시스템: 유튜브, 넷플릭스 등의 맞춤 콘텐츠 추천.\n이미지 및 음성 인식: 사진 속 사물 분류, 음성 인식 서비스.\n사기 탐지 및 보안: 금융 거래의 이상 징후 감지.\n언어 모델: 챗봇 및 자동 번역 서비스.\n예측 모델: 주식 트레이딩 및 에너지 부하 예측. \n\n머신

In [90]:
tfidf_cos_df.iloc[4].sort_values(ascending=False)[1:4] 

fname
C:/IT/workspace_python/LLM/lec05_data/g.txt    0.345135
C:/IT/workspace_python/LLM/lec05_data/f.txt    0.169286
C:/IT/workspace_python/LLM/lec05_data/h.txt    0.154259
Name: C:/IT/workspace_python/LLM/lec05_data/e.txt, dtype: float64

In [91]:
flist = tfidf_cos_df.iloc[4].sort_values(ascending=False)[1:4].index 
print(flist.values)
print("---"*20)
print( df[df['fname'].isin(flist)]['content'])   #.str[:30].values

['C:/IT/workspace_python/LLM/lec05_data/g.txt'
 'C:/IT/workspace_python/LLM/lec05_data/f.txt'
 'C:/IT/workspace_python/LLM/lec05_data/h.txt']
------------------------------------------------------------
5    머신 러닝의 주요 유형\n지도 학습 (Supervised Learning): 라벨링된(정답이 있는) 데이터를 통해 학습하여 입력값에 대한 예측값(분류/회귀)을 학습합니다.\n비지도 학습 (Unsupervised Learning): 라벨링되지 않은 데이터에서 숨겨진 구조나 패턴을 찾아냅니다.\n강화 학습 (Reinforcement Learning): 행동에 대한 보상을 통해 스스로 최적의 행동 방식을 학습합니다. \n\n머신 러닝 활용 예시\n추천 시스템: 유튜브, 넷플릭스 등의 맞춤 콘텐츠 추천.\n이미지 및 음성 인식: 사진 속 사물 분류, 음성 인식 서비스.\n사기 탐지 및 보안: 금융 거래의 이상 징후 감지.\n언어 모델: 챗봇 및 자동 번역 서비스.\n예측 모델: 주식 트레이딩 및 에너지 부하 예측. \n\n머신 러닝의 동의어 및 관련 용어\n기계 학습\nML (Machine Learning)\n데이터 마이닝 (부분집합)\n예측 분석 (Predictive Analytics) \n머신 러닝은 일반 프로그램이 고정된 규칙대로 움직이는 것과 달리, 데이터를 통해 경험을 쌓아 업무 수행 능력을 점진적으로 향상시키는 기술입니다.
6                                                   머신러닝(기계학습)은 데이터에서 규칙을 찾아 학습하는 인공지능의 한 분야이며, 딥러닝은 이를 발전시킨 인공신경망 기반의 하위 개념입니다. 가장 큰 차이는 사람의 개입 여부로, 머신러닝은 사람이 특징을 정의해줘야 하지만, 딥러닝은 인공지능이 스스로 데이터의 특징을 추출하여 학습합니다

| No | 100자 요약                                                               |
| -- | --------------------------------------------------------------------- |
| 5  | 머신러닝은 데이터 기반 학습 기술로 지도·비지도·강화 학습으로 구분되며 추천, 인식, 보안, 예측 등 다양한 분야에 활용된다 |
| 6  | 머신러닝은 사람이 특징을 정의하고 딥러닝은 스스로 학습하는 신경망 기반 기술로 데이터량과 활용 분야에서 차이가 있다      |
| 7  | 머신러닝은 데이터 패턴 학습 알고리즘으로 지도·비지도·강화 학습으로 나뉘며 회귀, 트리, SVM 등 다양한 기법이 사용된다  |
